### Scikit-Learn Deep Neural Network (MLP)
Since the Windows C++ environment is completely broken for TensorFlow and PyTorch, this is a pure CPU Deep Neural Network using Scikit-Learn.
It uses `partial_fit` to train on massive chunks of data without crashing your RAM.

In [ ]:
import os
import time
import numpy as np
from sklearn.neural_network import MLPClassifier
import joblib

try:
    from thrember.features import PEFeatureExtractor
    ndim = PEFeatureExtractor.dim
    print(f"Features extractor imported. Dimension: {ndim}")
except ImportError:
    ndim = 2381
    print(f"Warning: 'thrember' not found. Using default dimension: {ndim}")

In [ ]:
# --- CONFIGURATION ---
DATASET_DIR = r"Z:\ember2024_train_data"
CHUNK_SIZE = 250000              
EPOCHS = 10                      

checkpoint_path = os.path.join(DATASET_DIR, "ember_sklearn_dnn.pkl")

In [ ]:
# --- MEMORY MAPPING ---
X_path = os.path.join(DATASET_DIR, "X_train.dat")
y_path = os.path.join(DATASET_DIR, "y_train.dat")

file_size = os.path.getsize(X_path)
nrows = file_size // (ndim * 4)
train_nrows = int(nrows * 0.9)

X_memmap = np.memmap(X_path, dtype=np.float32, mode="r", shape=(nrows, ndim))
y_memmap = np.memmap(y_path, dtype=np.int32, mode="r", shape=(nrows,))

print(f"Total Rows: {nrows:,} | Training on {train_nrows:,} samples.")

In [ ]:
# --- BUILD DEEP NEURAL NETWORK ---
# A 3-layer Deep Neural Net
mlp = MLPClassifier(
    hidden_layer_sizes=(1024, 512, 128),
    activation='relu',
    solver='adam',
    alpha=0.0001,           # L2 Regularization (similar to Weight Decay)
    batch_size=4096,        # Process 4096 samples at a time in memory
    learning_rate='adaptive',
    max_iter=1,             # We handle epochs manually via partial_fit
    warm_start=True,        # Keep weights between partial_fits
    verbose=False,
    random_state=42
)

# Define the classes for the very first partial_fit call
classes = np.array([0, 1])

In [ ]:
# --- THE TRAINING LOOP ---
total_chunks = (train_nrows + CHUNK_SIZE - 1) // CHUNK_SIZE

print("Starting Scikit-Learn DNN Out-of-Core Batch Training...")

for epoch in range(EPOCHS):
    start_time = time.time()
    current_chunk = 0
    
    for start_idx in range(0, train_nrows, CHUNK_SIZE):
        end_idx = min(start_idx + CHUNK_SIZE, train_nrows)
        
        # Load EXACT chunk into RAM
        X_chunk = np.array(X_memmap[start_idx:end_idx])
        y_chunk = np.array(y_memmap[start_idx:end_idx])
        
        # Filter unlabeled (-1)
        valid_idx = y_chunk != -1
        X_chunk, y_chunk = X_chunk[valid_idx], y_chunk[valid_idx]
        
        if len(y_chunk) > 0:
            # Partial fit updates the weights using standard backpropagation
            mlp.partial_fit(X_chunk, y_chunk, classes=classes)
            
        current_chunk += 1
        print(f"  [E{epoch+1}] Chunk {current_chunk}/{total_chunks}: Processed up to row {end_idx:,}")
        
    elapsed = time.time() - start_time
    print(f"=> EPOCH {epoch+1} COMPLETE | Time: {elapsed:.2f} sec\n")
    
    # Save Checkpoint
    joblib.dump(mlp, checkpoint_path)
    print(f"Checkpoint saved to {checkpoint_path}\n")

print("\n🎉 Deep Learning Training Complete!")